# Automação do Processo de Análise das Lojas

Este notebook foi desenvolvido para automatizar o processo de análise das vendas das lojas.

Ao longo do processo, os dados são organizados, os principais indicadores são calculados, os relatórios são gerados e as informações são enviadas automaticamente por e-mail aos responsáveis.


## Carregamento dos dados

Primeiro, são carregadas as bases que serão utilizadas durante o processo.

A partir dessas informações, os dados de vendas são relacionados às suas respectivas lojas para os cálculos dos indicadores.


In [ ]:
# Importa as bibliotecas utilizadas

import pandas as pd
import pathlib
import base64
import os
from email.message import EmailMessage
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow

In [ ]:
# Carrega as bases de dados utilizadas
emails = pd.read_excel(r"Bases de Dados\Emails.xlsx")
lojas = pd.read_csv(r"Bases de Dados\Lojas.csv", encoding="utf-8", sep=";")
vendas = pd.read_excel(r"Bases de Dados\Vendas.xlsx")

# Exibe as bases para conferir os dados carregados
display(emails)
display(lojas)
display(vendas)

## Organização dos dados

Os dados de vendas são relacionados às informações das lojas e separados para facilitar os cálculos e a análise de cada unidade.

In [ ]:
# Relaciona as vendas com as informações das lojas

vendas = vendas.merge(lojas, on="ID Loja")

display(vendas)

In [ ]:
# Separa os dados de cada loja

dicionario_lojas = {}

for loja in lojas['Loja']:
    dicionario_lojas[loja] = vendas.loc[vendas['Loja'] == loja, :]


# Confere os dados de algumas lojas

display(dicionario_lojas['Rio Mar Recife'])
display(dicionario_lojas['Shopping Vila Velha'])

In [ ]:
# Identifica a data mais recente disponível nos dados

dia_indicador = vendas['Data'].max()

print(dia_indicador)
print('{}/{}'.format(dia_indicador.day, dia_indicador.month))

### Backup dos dados

Após organizar os dados, são criadas as pastas de cada loja e os arquivos de vendas são salvos como backup.

In [ ]:
# Verifica as pastas de backup existentes

caminho_backup = pathlib.Path(r'Backup Arquivos Lojas')

# Cria uma pasta para cada loja e salva os dados

caminho_backup.mkdir(exist_ok=True)

arquivos_pasta_backup = caminho_backup.iterdir()
lista_nomes_backup = [arquivo.name for arquivo in arquivos_pasta_backup]

for loja in dicionario_lojas:
    if loja not in lista_nomes_backup:
        nova_pasta = caminho_backup / loja
        nova_pasta.mkdir()
    
    # Define o nome e o local do arquivo

    nome_arquivo = '{}_{}_{}.xlsx'.format(
        dia_indicador.month,
        dia_indicador.day,
        loja
    )

    local_arquivo = caminho_backup / loja / nome_arquivo
    dicionario_lojas[loja].to_excel(local_arquivo)

## Indicadores

Nesta etapa, são definidas as metas e calculados os principais indicadores de desempenho das lojas.

In [ ]:
# Definição de metas

meta_faturamento_dia = 1000
meta_faturamento_ano = 1650000
meta_qtdeprodutos_dia = 4
meta_qtdeprodutos_ano = 120
meta_ticketmedio_dia = 500
meta_ticketmedio_ano = 500

### Conexão com o Gmail

A conexão com o Gmail permite que os relatórios sejam enviados automaticamente para os responsáveis por cada loja.

In [ ]:
# Define as permissões utilizadas para acessar o Gmail

SCOPES = [
    "https://www.googleapis.com/auth/gmail.send"
]

creds = None

# Verifica se já existe uma autenticação salva

if os.path.exists("token.json"):
    creds = Credentials.from_authorized_user_file(
        "token.json",
        SCOPES
    )

# Realiza a autenticação caso necessário

if not creds or not creds.valid:

    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())

    else:
        flow = InstalledAppFlow.from_client_secrets_file(
            "credentials.json",
            SCOPES
        )

        creds = flow.run_local_server(port=0)

    # Salva a autenticação para os próximos acessos

    with open("token.json", "w") as token:
        token.write(creds.to_json())

print("Gmail autenticado com sucesso!")

In [ ]:
# Conecta o código à API do Gmail

from googleapiclient.discovery import build

service = build(
    "gmail",
    "v1",
    credentials=creds
)

print("API Gmail conectada!")

### Cálculo dos indicadores

Os dados de cada loja são analisados para calcular o faturamento, a quantidade de produtos e o ticket médio, comparando os resultados com as metas definidas.

In [ ]:
# Calcula os indicadores de cada loja

for loja in dicionario_lojas:

    vendas_loja = dicionario_lojas[loja]

    # Filtra as vendas do dia analisado

    vendas_loja_dia = vendas_loja.loc[
        vendas_loja['Data'] == dia_indicador,
        :
    ]


    # Calcula o faturamento

    faturamento_ano = vendas_loja['Valor Final'].sum()
    faturamento_dia = vendas_loja_dia['Valor Final'].sum()


    # Calcula a quantidade de produtos

    qtde_produtos_ano = len(
        vendas_loja['Produto'].unique()
    )

    qtde_produtos_dia = len(
        vendas_loja_dia['Produto'].unique()
    )


    # Calcula o ticket médio

    valor_venda = vendas_loja.groupby(
        'Código Venda'
    )['Valor Final'].sum()

    ticket_medio_ano = valor_venda.mean()


    # Calcula o ticket médio do dia

    valor_venda_dia = vendas_loja_dia.groupby(
        'Código Venda'
    )['Valor Final'].sum()

    ticket_medio_dia = valor_venda_dia.mean()


    # Busca as informações do gerente

    nome = emails.loc[
        emails['Loja'] == loja,
        'Gerente'
    ].values[0]

    destinatario = emails.loc[
        emails['Loja'] == loja,
        'E-mail'
    ].values[0]


    # Define o assunto do e-mail

    assunto = f'OnePage Dia {dia_indicador.day}/{dia_indicador.month} - Loja {loja}'


    # Define a cor dos indicadores de acordo com as metas

    if faturamento_dia >= meta_faturamento_dia:
        cor_fat_dia = 'green'
    else:
        cor_fat_dia = 'red'

    if faturamento_ano >= meta_faturamento_ano:
        cor_fat_ano = 'green'
    else:
        cor_fat_ano = 'red'

    if qtde_produtos_dia >= meta_qtdeprodutos_dia:
        cor_qtde_dia = 'green'
    else:
        cor_qtde_dia = 'red'

    if qtde_produtos_ano >= meta_qtdeprodutos_ano:
        cor_qtde_ano = 'green'
    else:
        cor_qtde_ano = 'red'

    if ticket_medio_dia >= meta_ticketmedio_dia:
        cor_ticket_dia = 'green'
    else:
        cor_ticket_dia = 'red'

    if ticket_medio_ano >= meta_ticketmedio_ano:
        cor_ticket_ano = 'green'
    else:
        cor_ticket_ano = 'red'


    # Monta o conteúdo do e-mail

    html = f'''
    <p>Bom dia, {nome}</p>

    <p>O resultado de ontem <strong>({dia_indicador.day}/{dia_indicador.month})</strong> da <strong>Loja {loja}</strong> foi:</p>

    <table>
      <tr>
        <th>Indicador</th>
        <th>Valor Dia</th>
        <th>Meta Dia</th>
        <th>Cenário Dia</th>
      </tr>

      <tr>
        <td>Faturamento</td>
        <td style="text-align: center">R${faturamento_dia:.2f}</td>
        <td style="text-align: center">R${meta_faturamento_dia:.2f}</td>
        <td style="text-align: center">
            <font color="{cor_fat_dia}">◙</font>
        </td>
      </tr>

      <tr>
        <td>Diversidade de Produtos</td>
        <td style="text-align: center">{qtde_produtos_dia}</td>
        <td style="text-align: center">{meta_qtdeprodutos_dia}</td>
        <td style="text-align: center">
            <font color="{cor_qtde_dia}">◙</font>
        </td>
      </tr>

      <tr>
        <td>Ticket Médio</td>
        <td style="text-align: center">R${ticket_medio_dia:.2f}</td>
        <td style="text-align: center">R${meta_ticketmedio_dia:.2f}</td>
        <td style="text-align: center">
            <font color="{cor_ticket_dia}">◙</font>
        </td>
      </tr>
    </table>

    <br>

    <table>
      <tr>
        <th>Indicador</th>
        <th>Valor Ano</th>
        <th>Meta Ano</th>
        <th>Cenário Ano</th>
      </tr>

      <tr>
        <td>Faturamento</td>
        <td style="text-align: center">R${faturamento_ano:.2f}</td>
        <td style="text-align: center">R${meta_faturamento_ano:.2f}</td>
        <td style="text-align: center">
            <font color="{cor_fat_ano}">◙</font>
        </td>
      </tr>

      <tr>
        <td>Diversidade de Produtos</td>
        <td style="text-align: center">{qtde_produtos_ano}</td>
        <td style="text-align: center">{meta_qtdeprodutos_ano}</td>
        <td style="text-align: center">
            <font color="{cor_qtde_ano}">◙</font>
        </td>
      </tr>

      <tr>
        <td>Ticket Médio</td>
        <td style="text-align: center">R${ticket_medio_ano:.2f}</td>
        <td style="text-align: center">R${meta_ticketmedio_ano:.2f}</td>
        <td style="text-align: center">
            <font color="{cor_ticket_ano}">◙</font>
        </td>
      </tr>
    </table>

    <p>Segue em anexo a planilha com todos os dados para mais detalhes.</p>

    <p>Qualquer dúvida estou à disposição.</p>

    <p>Att., Lira</p>
    '''


    # Localiza a planilha que será enviada

    attachment = (
        pathlib.Path.cwd()
        / caminho_backup
        / loja
        / f'{dia_indicador.month}_{dia_indicador.day}_{loja}.xlsx'
    )


    # Cria a mensagem de e-mail

    mensagem = EmailMessage()

    mensagem["To"] = destinatario
    mensagem["Subject"] = assunto

    mensagem.add_alternative(
        html,
        subtype='html'
    )


    # Adiciona a planilha como anexo

    with open(attachment, 'rb') as arquivo:

        mensagem.add_attachment(
            arquivo.read(),
            maintype='application',
            subtype='vnd.openxmlformats-officedocument.spreadsheetml.sheet',
            filename=attachment.name
        )


    # Prepara a mensagem para envio

    encoded_message = base64.urlsafe_b64encode(
        mensagem.as_bytes()
    ).decode()


    # Envia o e-mail

    resultado = service.users().messages().send(
        userId='me',
        body={
            'raw': encoded_message
        }
    ).execute()


    print(
        'E-mail da Loja {} enviado'.format(loja)
    )

## Ranking das lojas

Nesta etapa, as lojas são organizadas de acordo com o faturamento para gerar um ranking anual e outro referente ao dia analisado.

In [ ]:
# Calcula o faturamento anual de cada loja

faturamento_lojas = vendas.groupby('Loja')['Valor Final'].sum()


# Ordena as lojas da maior para a menor

faturamento_lojas_ano = faturamento_lojas.sort_values(ascending=False)

display(faturamento_lojas_ano)


# Salva o ranking anual

nome_arquivo = '{}_{}_Ranking Anual.xlsx'.format(
    dia_indicador.month,
    dia_indicador.day
)

faturamento_lojas_ano.to_excel(
    r'Backup Arquivos Lojas\{}'.format(nome_arquivo)
)


# Filtra as vendas do dia analisado

vendas_dia = vendas.loc[vendas['Data'] == dia_indicador, :]


# Calcula o faturamento diário de cada loja

faturamento_lojas_dia = vendas_dia.groupby('Loja')['Valor Final'].sum()


# Ordena as lojas da maior para a menor

faturamento_lojas_dia = faturamento_lojas_dia.sort_values(ascending=False)

display(faturamento_lojas_dia)


# Salva o ranking do dia

nome_arquivo = '{}_{}_Ranking Dia.xlsx'.format(
    dia_indicador.month,
    dia_indicador.day
)

faturamento_lojas_dia.to_excel(
    r'Backup Arquivos Lojas\{}'.format(nome_arquivo)
)

In [ ]:
#envia o e-mail para a Diretoria

destinatario = emails.loc[
    emails['Loja'] == 'Diretoria',
    'E-mail'
].values[0]

assunto = f'Ranking Dia {dia_indicador.day}/{dia_indicador.month}'

# Corpo do e-mail
corpo = f'''
Prezados, bom dia

Melhor loja do Dia em Faturamento: Loja {faturamento_lojas_dia.index[0]} com Faturamento R${faturamento_lojas_dia.iloc[0]:.2f}

Pior loja do Dia em Faturamento: Loja {faturamento_lojas_dia.index[-1]} com Faturamento R${faturamento_lojas_dia.iloc[-1]:.2f}

Melhor loja do Ano em Faturamento: Loja {faturamento_lojas_ano.index[0]} com Faturamento R${faturamento_lojas_ano.iloc[0]:.2f}

Pior loja do Ano em Faturamento: Loja {faturamento_lojas_ano.index[-1]} com Faturamento R${faturamento_lojas_ano.iloc[-1]:.2f}

Segue em anexo os rankings do ano e do dia de todas as lojas.

Qualquer dúvida estou à disposição.

Att.,
Lira
'''

# Cria a mensagem
mensagem = EmailMessage()

mensagem["To"] = destinatario
mensagem["Subject"] = assunto

# Adiciona o corpo do e-mail
mensagem.set_content(corpo)

# Anexos
anexos = [
    pathlib.Path.cwd()
    / caminho_backup
    / f'{dia_indicador.month}_{dia_indicador.day}_Ranking Anual.xlsx',

    pathlib.Path.cwd()
    / caminho_backup
    / f'{dia_indicador.month}_{dia_indicador.day}_Ranking Dia.xlsx'
]

# Adicionar os anexos ao e-mail
for attachment in anexos:
    with open(attachment, 'rb') as arquivo:
        mensagem.add_attachment(
            arquivo.read(),
            maintype='application',
            subtype='vnd.openxmlformats-officedocument.spreadsheetml.sheet',
            filename=attachment.name
        )

# Codifica a mensagem para envio pela API do Gmail
encoded_message = base64.urlsafe_b64encode(
    mensagem.as_bytes()
).decode()

# Enviar o e-mail
resultado = service.users().messages().send(
    userId='me',
    body={
        'raw': encoded_message
    }
).execute()

print('E-mail da Diretoria enviado pelo Gmail')